# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**xie12346789**  
第5天专题（A/B/C/D/E）：**G01**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "xie12346789"
TOPIC = "G01"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [ ]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


In [ ]:
business_questions = {
    "category_bar": "不同偏好订单类别用户的流失率有何差异？",
    "behavior_scatter": "用户订单数量与返现金额之间存在何种关系？",
    "ordered_line": "不同使用年限阶段的用户流失率变化趋势如何？",
    "composition_chart": "用户偏好的支付方式分布情况如何？",
}

chart_reasons = {
    "category_bar": "使用柱状图对比不同订单类别的流失率，便于直观比较各组间差异，同时标注样本量确保可比性。",
    "behavior_scatter": "使用散点图展示订单数量与返现金额的关系，通过颜色区分流失状态，便于发现数据分布模式和异常值。",
    "ordered_line": "使用折线图展示按使用年限分组的流失率变化，TenureGroup是有序阶段，适合折线图展示趋势。",
    "composition_chart": "支付方式类别较少（5个以内），适合使用饼图展示整体构成比例。",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")


## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
category_field = "PreferedOrderCat"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
category_summary = category_summary.sort_values(by="流失率", ascending=False)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)


In [ ]:
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

bars = ax_bar.bar(category_summary[category_field], category_summary["流失率"] * 100, color="#4A90D9")

for bar, row in zip(bars, category_summary.itertuples()):
    height = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width() / 2, height,
                f"{height:.1f}%\n(n={row.用户数})",
                ha="center", va="bottom", fontsize=10)

ax_bar.set_title("不同偏好订单类别用户流失率对比", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("偏好订单类别", fontsize=12)
ax_bar.set_ylabel("流失率 (%)", fontsize=12)
ax_bar.set_ylim(0, 35)
ax_bar.grid(axis="y", alpha=0.3)

plt.xticks(rotation=30, ha="right")

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))


### 柱状图结论

- 观察：Electronics类别用户流失率最高，Laptop类别用户流失率最低。
- 证据：Electronics类别流失率20.4%（n=1696），Laptop类别流失率仅6.6%（n=303），差距约13.8个百分点。Fashion类别流失率15.6%（n=1064），Grocery类别流失率14.5%（n=2035）。
- 边界：该图仅展示流失率与订单类别的关联，不能证明因果关系；Laptop样本量较小，结论可靠性需谨慎对待。


## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]

ax_scatter.scatter(churn_0[x_field], churn_0[y_field], 
                   color="#4A90D9", alpha=0.5, label="未流失", s=30)
ax_scatter.scatter(churn_1[x_field], churn_1[y_field], 
                   color="#E74C3C", alpha=0.5, label="已流失", s=30)

ax_scatter.set_title("用户订单数量与返现金额关系图", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel("订单数量", fontsize=12)
ax_scatter.set_ylabel("返现金额 (元)", fontsize=12)
ax_scatter.legend(fontsize=12)
ax_scatter.grid(True, alpha=0.3)

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))


### 散点图结论

- 观察：订单数量与返现金额呈正相关关系，未流失用户在高订单量区域分布更集中。
- 证据：数据点从左下角向右上角倾斜，说明订单越多返现越高；已流失用户（红色）在低订单量区域分布较多，高订单量区域较少。
- 边界：相关关系不等于因果关系，返现金额高可能是订单多的结果而非原因；数据存在部分异常值。


## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["0个月", "1-12个月", "13-24个月", "25-36个月", "37-48个月", "49-60个月", "61-72个月"]

ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
ordered_summary[ordered_field] = pd.Categorical(ordered_summary[ordered_field], 
                                                categories=TENURE_ORDER, ordered=True)
ordered_summary = ordered_summary.sort_values(by=ordered_field)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)


In [ ]:
fig_line, ax_line = plt.subplots(figsize=(10, 6))

ax_line.plot(ordered_summary[ordered_field], ordered_summary["流失率"] * 100, 
             marker="o", color="#4A90D9", linewidth=2, markersize=8)

for x, y, n in zip(ordered_summary[ordered_field], 
                   ordered_summary["流失率"] * 100, 
                   ordered_summary["用户数"]):
    ax_line.text(x, y, f"{y:.1f}%\n(n={n})", 
                 ha="center", va="bottom", fontsize=9)

ax_line.set_title("不同使用年限阶段用户流失率趋势", fontsize=14, fontweight="bold")
ax_line.set_xlabel("使用年限阶段", fontsize=12)
ax_line.set_ylabel("流失率 (%)", fontsize=12)
ax_line.set_ylim(0, 25)
ax_line.grid(True, alpha=0.3)

plt.xticks(rotation=30, ha="right")

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))


### 折线图结论

- 观察：新用户（0个月）流失率最高，随着使用年限增加流失率逐渐下降，49-60个月达到最低点后略有上升。
- 证据：0个月用户流失率20.0%（n=60），1-12个月16.2%（n=745），49-60个月降至11.3%（n=746），之后61-72个月回升至15.8%（n=764）。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势；0个月样本量较小（n=60），结论稳定性有限。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
composition_field = "PreferredPaymentMode"
composition_summary = (
    df.groupby(composition_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary = composition_summary.sort_values(by="占比", ascending=False)

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)


In [ ]:
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))

colors = ["#4A90D9", "#67B8DC", "#95D2EA", "#B8E4F0", "#E0F4FF"]
wedges, texts, autotexts = ax_composition.pie(
    composition_summary["占比"],
    labels=composition_summary[composition_field],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"width": 0.4},
    textprops={"fontsize": 11}
)

ax_composition.set_title("用户偏好支付方式分布", fontsize=14, fontweight="bold")

legend_labels = [f"{row[composition_field]} ({row['用户数']}人)"
                for _, row in composition_summary.iterrows()]
ax_composition.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=10)

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))


### 构成图结论

- 观察：Cash on Delivery是最受欢迎的支付方式，占比超过40%，其次是Debit Card和E-wallet。
- 证据：Cash on Delivery占比41.1%（2314人），Debit Card占比24.4%（1372人），E-wallet占比22.5%（1267人），Credit Card占比9.5%（535人），UPI占比2.5%（142人）。
- 边界：该图适合展示各类别占比，但不适合比较不同类别的流失率等其他指标；类别数量较少（5个），适合饼图展示。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(category_summary[category_field], category_summary["流失率"] * 100, color="#4A90D9")
axes[0, 0].set_title("订单类别流失率", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("流失率 (%)")
axes[0, 0].tick_params(axis="x", rotation=30, labelsize=9)
axes[0, 0].grid(axis="y", alpha=0.3)

axes[0, 1].scatter(churn_0[x_field], churn_0[y_field], color="#4A90D9", alpha=0.3, s=20)
axes[0, 1].scatter(churn_1[x_field], churn_1[y_field], color="#E74C3C", alpha=0.3, s=20)
axes[0, 1].set_title("订单数与返现关系", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("订单数量")
axes[0, 1].set_ylabel("返现金额")
axes[0, 1].legend(["未流失", "已流失"], fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(ordered_summary[ordered_field], ordered_summary["流失率"] * 100, 
                marker="o", color="#4A90D9", linewidth=2)
axes[1, 0].set_title("使用年限流失趋势", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("使用年限阶段")
axes[1, 0].set_ylabel("流失率 (%)")
axes[1, 0].tick_params(axis="x", rotation=30, labelsize=9)
axes[1, 0].grid(True, alpha=0.3)

colors = ["#4A90D9", "#67B8DC", "#95D2EA", "#B8E4F0", "#E0F4FF"]
axes[1, 1].pie(composition_summary["占比"], labels=composition_summary[composition_field],
               autopct="%1.1f%%", startangle=90, colors=colors, textprops={"fontsize": 9},
               wedgeprops={"width": 0.4})
axes[1, 1].set_title("支付方式分布", fontsize=12, fontweight="bold")

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))


## 综合发现与局限

1. 综合发现1：电子产品类用户流失率最高（20.4%），而笔记本电脑类用户流失率最低（6.6%），说明不同产品类别的用户忠诚度存在显著差异。
2. 综合发现2：新用户（0个月）流失率高达20%，远高于其他阶段，表明用户初次使用体验对留存至关重要。
3. 综合发现3：货到付款（Cash on Delivery）是最主流的支付方式（41.1%），其次是借记卡（24.4%）和电子钱包（22.5%），建议优化货到付款用户的服务体验。
4. 数据或方法局限：CashbackAmount是返现金额，不是销售额、收入或GMV；数据为快照数据而非时间序列，无法观察真实的时间趋势；部分类别样本量较小可能影响结论稳定性。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png", 
     "business_question": "不同偏好订单类别用户的流失率有何差异？", 
     "chart_type": "bar", 
     "key_finding": "Electronics类别流失率最高(20.4%)，Laptop类别最低(6.6%)，各类别流失率差异显著", 
     "limitation": "仅展示关联关系，不能证明因果；Laptop样本量较小(n=303)，结论可靠性需谨慎"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png", 
     "business_question": "用户订单数量与返现金额之间存在何种关系？", 
     "chart_type": "scatter", 
     "key_finding": "订单数量与返现金额呈正相关，未流失用户在高订单量区域分布更集中", 
     "limitation": "相关关系不等于因果关系；存在部分异常值影响视觉判断"},
    {"chart_id": "03", "file_name": "03_ordered_line.png", 
     "business_question": "不同使用年限阶段的用户流失率变化趋势如何？", 
     "chart_type": "line", 
     "key_finding": "新用户流失率最高(20%)，随使用年限增加流失率下降，49-60个月达最低点(11.3%)", 
     "limitation": "是有序阶段比较，非真实时间趋势；0个月样本量小(n=60)影响稳定性"},
    {"chart_id": "04", "file_name": "04_composition_chart.png", 
     "business_question": "用户偏好的支付方式分布情况如何？", 
     "chart_type": "pie", 
     "key_finding": "Cash on Delivery占比最高(41.1%)，其次是Debit Card(24.4%)和E-wallet(22.5%)", 
     "limitation": "适合展示构成比例，但不适合比较流失率等其他指标"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png", 
     "business_question": "整体概览", 
     "chart_type": "dashboard", 
     "key_finding": "电子产品用户流失风险最高，新用户留存是重点，货到付款用户占比最大", 
     "limitation": "综合图为概览性质，详细数值需参考独立图表"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)


In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
